# 🔍 Project 3: Machine Learning System for Fraud Detection
### RISE Internship — Machine Learning & AI | Tamizhan Skills

---

## 📌 Problem Statement
Financial institutions and e-commerce platforms face increasing fraudulent transactions that result in major financial losses. Rule-based systems fail to detect evolving fraud patterns. Industry relies on machine learning models to identify suspicious activities in real time.

## 🎯 Objective
To build a machine learning model that detects fraudulent transactions based on transaction behavior and historical data.

## 🛠️ Tools Used
- Python, NumPy, Pandas, Matplotlib, Seaborn, Scikit-learn, Jupyter Notebook

---

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score
)
from sklearn.utils import resample
import os
os.makedirs('outputs', exist_ok=True)

print('✅ All libraries imported successfully!')

---
## Step 2: Transaction Dataset Ingestion
> We create a realistic synthetic dataset of 10,000 transactions with a 5% fraud rate — this mimics real-world class imbalance seen in actual fraud datasets (like Kaggle's Credit Card Fraud dataset).

In [ ]:
np.random.seed(42)
n_samples = 10000
n_fraud   = 500   # 5% fraud rate — realistic imbalance

# ── Legitimate transactions ──────────────────────────────────
legit_df = pd.DataFrame({
    'Time'            : np.random.uniform(0, 172800,    size=n_samples - n_fraud),
    'Amount'          : np.random.exponential(80,       size=n_samples - n_fraud),
    'V1'              : np.random.normal(0,   1,        size=n_samples - n_fraud),
    'V2'              : np.random.normal(0,   1,        size=n_samples - n_fraud),
    'V3'              : np.random.normal(0,   1,        size=n_samples - n_fraud),
    'V4'              : np.random.normal(0,   1,        size=n_samples - n_fraud),
    'V5'              : np.random.normal(0,   1,        size=n_samples - n_fraud),
    'MerchantCategory': np.random.choice(['retail','food','travel','entertainment','medical'],
                                         size=n_samples - n_fraud),
    'Class'           : 0
})
legit_df['Hour'] = (legit_df['Time'] / 3600 % 24).astype(int)

# ── Fraudulent transactions (different statistical profile) ──
fraud_df = pd.DataFrame({
    'Time'            : np.random.uniform(0, 172800,    size=n_fraud),
    'Amount'          : np.random.exponential(400,      size=n_fraud),
    'V1'              : np.random.normal(-2, 1.5,       size=n_fraud),  # lower
    'V2'              : np.random.normal( 3, 2,         size=n_fraud),  # higher
    'V3'              : np.random.normal(-1, 1,         size=n_fraud),
    'V4'              : np.random.normal( 2, 1.5,       size=n_fraud),
    'V5'              : np.random.normal(-3, 2,         size=n_fraud),  # much lower
    'MerchantCategory': np.random.choice(['travel','entertainment'], size=n_fraud),
    'Class'           : 1
})
fraud_df['Hour'] = (fraud_df['Time'] / 3600 % 24).astype(int)

# ── Combine and shuffle ──────────────────────────────────────
df = pd.concat([legit_df, fraud_df], ignore_index=True)\
       .sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset shape   : {df.shape}')
print(f'Legit txns      : {(df["Class"]==0).sum()}')
print(f'Fraud txns      : {(df["Class"]==1).sum()}')
print(f'Fraud rate      : {df["Class"].mean()*100:.2f}%')
df.head()

---
## Step 3: Exploratory Data Analysis (EDA)
> Understand the data distribution, class imbalance, and key fraud patterns before modelling.

In [ ]:
# Basic statistics
print('=== Dataset Info ===')
print(df.info())
print('\n=== Statistical Summary ===')
df.describe()

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nClass distribution:')
print(df['Class'].value_counts())

In [ ]:
# ── EDA Plot 1: Overview ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('EDA — Transaction Overview', fontsize=14, fontweight='bold')

# Class balance
class_counts = df['Class'].value_counts()
bars = axes[0].bar(['Legitimate', 'Fraud'], class_counts.values,
                   color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 50, str(val), ha='center', fontweight='bold')

# Amount by class
df[df['Class']==0]['Amount'].hist(bins=50, ax=axes[1], alpha=0.7, color='#2ecc71', label='Legit')
df[df['Class']==1]['Amount'].hist(bins=50, ax=axes[1], alpha=0.7, color='#e74c3c', label='Fraud')
axes[1].set_title('Transaction Amount Distribution')
axes[1].set_xlabel('Amount ($)')
axes[1].legend()

# Fraud by hour
fraud_by_hour = df[df['Class']==1].groupby('Hour').size()
axes[2].bar(fraud_by_hour.index, fraud_by_hour.values, color='#e74c3c', alpha=0.8)
axes[2].set_title('Fraud Transactions by Hour of Day')
axes[2].set_xlabel('Hour')
axes[2].set_ylabel('Fraud Count')

plt.tight_layout()
plt.savefig('outputs/01_eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── EDA Plot 2: Feature distributions ───────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('EDA — Feature Distributions by Class', fontsize=14, fontweight='bold')

for i, feat in enumerate(['V1', 'V2', 'V3', 'V4', 'V5', 'Amount']):
    ax = axes[i//3][i%3]
    df[df['Class']==0][feat].hist(bins=40, ax=ax, alpha=0.6, color='#2ecc71', label='Legit', density=True)
    df[df['Class']==1][feat].hist(bins=40, ax=ax, alpha=0.6, color='#e74c3c', label='Fraud', density=True)
    ax.set_title(f'{feat} — Legit vs Fraud')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('outputs/02_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 4: Data Cleaning & Preprocessing
> Encode categorical features and normalize numerical ones.

In [ ]:
# One-hot encode MerchantCategory
df_encoded = pd.get_dummies(df, columns=['MerchantCategory'], drop_first=False)

# Features and target
X = df_encoded.drop('Class', axis=1).copy()
y = df_encoded['Class']

# Normalize Amount and Time
scaler = StandardScaler()
X[['Amount', 'Time']] = scaler.fit_transform(X[['Amount', 'Time']])

print(f'Features after encoding : {X.shape[1]}')
print(f'Feature list:\n{list(X.columns)}')
X.head()

---
## Step 5: Handling Class Imbalance
> **Key concept:** With only 5% fraud, a lazy model could just predict "legit" always and get 95% accuracy — yet catch zero fraud. We fix this by oversampling the minority (fraud) class, but ONLY in the training set. The test set must remain untouched to give honest evaluation.

In [ ]:
# Train/test split FIRST — before any resampling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size : {X_train.shape[0]}  |  Test size : {X_test.shape[0]}')
print(f'Train fraud: {int(y_train.sum())} ({y_train.mean()*100:.1f}%)')

# Oversample minority class in training data ONLY
train_df   = pd.concat([X_train, y_train], axis=1)
majority   = train_df[train_df['Class'] == 0]
minority   = train_df[train_df['Class'] == 1]

minority_upsampled = resample(minority, replace=True,
                               n_samples=len(majority), random_state=42)
train_balanced = pd.concat([majority, minority_upsampled])\
                   .sample(frac=1, random_state=42)

X_train_bal = train_balanced.drop('Class', axis=1)
y_train_bal = train_balanced['Class']

print(f'\nAfter oversampling — Train size : {X_train_bal.shape[0]}')
print(f'Balanced fraud rate             : {y_train_bal.mean()*100:.1f}%')

# Visualise before vs after
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['Legit','Fraud'], [int((y_train==0).sum()), int((y_train==1).sum())],
            color=['#2ecc71','#e74c3c'])
axes[0].set_title('Before Oversampling (Train)')
axes[1].bar(['Legit','Fraud'], [int((y_train_bal==0).sum()), int((y_train_bal==1).sum())],
            color=['#2ecc71','#e74c3c'])
axes[1].set_title('After Oversampling (Train)')
plt.suptitle('Class Imbalance Handling', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 6: Feature Engineering for Fraud Indicators
> We create new features that capture interactions between variables — fraud patterns often emerge from combinations, not single features.

In [ ]:
def add_fraud_features(X_df):
    X_df = X_df.copy()
    X_df['V1_V2_interaction'] = X_df['V1'] * X_df['V2']   # Combined signal
    X_df['V4_V5_interaction'] = X_df['V4'] * X_df['V5']   # Combined signal
    X_df['amount_v1_ratio']   = X_df['Amount'] / (X_df['V1'].abs() + 1e-6)  # Amount anomaly
    X_df['v_sum']             = X_df[['V1','V2','V3','V4','V5']].sum(axis=1)
    X_df['v_std']             = X_df[['V1','V2','V3','V4','V5']].std(axis=1)
    return X_df

X_train_bal = add_fraud_features(X_train_bal)
X_test      = add_fraud_features(X_test)

print(f'Final feature count: {X_train_bal.shape[1]}')
print(f'New features added : V1_V2_interaction, V4_V5_interaction, amount_v1_ratio, v_sum, v_std')

---
## Step 7: Machine Learning Model Training
> We train 4 classifiers and compare them. Each has its own strengths — Logistic Regression is interpretable, Random Forest is robust, Gradient Boosting often wins on tabular data.

In [ ]:
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    print(f'Training {name}...', end=' ')
    model.fit(X_train_bal, y_train_bal)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'model'     : model,
        'y_pred'    : y_pred,
        'y_prob'    : y_prob,
        'precision' : precision_score(y_test, y_pred),
        'recall'    : recall_score(y_test, y_pred),
        'f1'        : f1_score(y_test, y_pred),
        'accuracy'  : accuracy_score(y_test, y_pred),
        'roc_auc'   : roc_auc_score(y_test, y_prob),
    }
    print(f'✅  F1={results[name]["f1"]:.4f}  ROC-AUC={results[name]["roc_auc"]:.4f}')

---
## Step 8: Model Evaluation — Precision, Recall, F1-Score
> **Why these metrics?** Accuracy is misleading on imbalanced data. We use:
> - **Precision** — Of all flagged fraud, how many were real?
> - **Recall** — Of all real fraud, how many did we catch? *(most critical)*
> - **F1-Score** — Harmonic mean of Precision & Recall
> - **ROC-AUC** — Overall ranking ability of the model

In [ ]:
# Summary table
summary = pd.DataFrame({
    name: [v['precision'], v['recall'], v['f1'], v['accuracy'], v['roc_auc']]
    for name, v in results.items()
}, index=['Precision','Recall','F1-Score','Accuracy','ROC-AUC']).T

print('=== Model Comparison ===')
print(summary.round(4).to_string())

best_name = max(results, key=lambda k: results[k]['f1'])
best = results[best_name]
print(f'\n🏆 Best Model (by F1): {best_name}')

In [ ]:
# Full classification report for best model
print(f'=== Classification Report — {best_name} ===')
print(classification_report(y_test, best['y_pred'], target_names=['Legit','Fraud']))

In [ ]:
# ── Metrics comparison bar chart ─────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(summary))
width = 0.18
colors = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6']
for i, (metric, color) in enumerate(zip(['Precision','Recall','F1-Score','ROC-AUC'], colors)):
    ax.bar(x + i*width, summary[metric], width, label=metric, color=color, alpha=0.85)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(summary.index, rotation=15, ha='right')
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Precision, Recall, F1, ROC-AUC', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/03_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrices for all models ───────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Confusion Matrices — All Models', fontsize=13, fontweight='bold')

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=ax,
                xticklabels=['Legit','Fraud'], yticklabels=['Legit','Fraud'])
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('outputs/04_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 9: Visualization of Fraud Patterns

In [ ]:
# ── ROC Curves ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
for (name, res), color in zip(results.items(), ['#e74c3c','#3498db','#2ecc71','#9b59b6']):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['roc_auc']:.3f})", lw=2, color=color)

ax.plot([0,1],[0,1],'k--', lw=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/05_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Precision-Recall Curve (best model) ──────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
prec_vals, rec_vals, _ = precision_recall_curve(y_test, best['y_prob'])
ap = average_precision_score(y_test, best['y_prob'])
ax.plot(rec_vals, prec_vals, color='#e74c3c', lw=2, label=f'AP = {ap:.3f}')
ax.fill_between(rec_vals, prec_vals, alpha=0.15, color='#e74c3c')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'Precision-Recall Curve — {best_name}', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/06_precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Importance (Random Forest) ───────────────────────
rf_model = results['Random Forest']['model']
feat_imp = pd.Series(rf_model.feature_importances_, index=X_train_bal.columns)
top15    = feat_imp.nlargest(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors_imp = ['#e74c3c' if i < 5 else '#3498db' for i in range(len(top15))]
top15.sort_values().plot(kind='barh', ax=ax, color=colors_imp[::-1], edgecolor='black', linewidth=0.5)
ax.set_title('Top 15 Feature Importances — Random Forest', fontweight='bold')
ax.set_xlabel('Importance Score')
ax.axvline(x=top15.mean(), color='black', linestyle='--', alpha=0.5, label='Mean')
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/07_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fraud Pattern Correlation Heatmaps ───────────────────────
numeric_features = ['V1','V2','V3','V4','V5','Amount','Hour']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Fraud Pattern Analysis — Correlation Heatmaps', fontweight='bold', fontsize=13)

sns.heatmap(df[df['Class']==1][numeric_features].corr(),
            annot=True, fmt='.2f', cmap='Reds', ax=axes[0], linewidths=0.5)
axes[0].set_title('Fraudulent Transactions')

sns.heatmap(df[df['Class']==0][numeric_features].corr(),
            annot=True, fmt='.2f', cmap='Blues', ax=axes[1], linewidths=0.5)
axes[1].set_title('Legitimate Transactions')

plt.tight_layout()
plt.savefig('outputs/08_fraud_patterns_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 10: Prediction of Suspicious Transactions
> Flag real-world transactions with fraud probability > 70% for analyst review.

In [ ]:
y_prob_all = best['model'].predict_proba(X_test)[:, 1]
y_pred_all = best['model'].predict(X_test)

suspicious_df = X_test[['Amount','V1','V2','V3','V4','V5']].copy()
suspicious_df['True_Label']    = y_test.values
suspicious_df['Fraud_Prob']    = y_prob_all
suspicious_df['Predicted']     = y_pred_all
suspicious_df['Is_Suspicious'] = suspicious_df['Fraud_Prob'] > 0.7

print(f'Total flagged as suspicious (prob > 0.70) : {suspicious_df["Is_Suspicious"].sum()}')
print(f'Of those, confirmed actual fraud          : {int(suspicious_df[suspicious_df["Is_Suspicious"]=="True"]["True_Label"].sum() if False else suspicious_df[suspicious_df["Is_Suspicious"]==True]["True_Label"].sum())}')

print('\nTop 10 Most Suspicious Transactions:')
suspicious_df.sort_values('Fraud_Prob', ascending=False).head(10)

In [ ]:
# Save suspicious transactions to CSV
suspicious_df.sort_values('Fraud_Prob', ascending=False)\
             .to_csv('outputs/suspicious_transactions.csv', index=False)
print('✅ Saved: outputs/suspicious_transactions.csv')

---
## Step 11: Model Documentation & Final Summary

In [ ]:
print('=' * 60)
print('   FINAL MODEL SUMMARY')
print('=' * 60)
print(f'  Best Model     : {best_name}')
print(f'  Precision      : {best["precision"]:.4f}')
print(f'  Recall         : {best["recall"]:.4f}')
print(f'  F1-Score       : {best["f1"]:.4f}')
print(f'  ROC-AUC        : {best["roc_auc"]:.4f}')
print()
print('  Output files saved in: outputs/')
for f in sorted(os.listdir('outputs')):
    print(f'    - {f}')
print('=' * 60)
print('   PROJECT 3 COMPLETE ✓')
print('=' * 60)

---
## 📊 Conclusion

| Metric | Value |
|--------|-------|
| Best Model | Gradient Boosting |
| Precision | 0.9524 |
| Recall | 1.0000 |
| F1-Score | 0.9756 |
| ROC-AUC | 1.0000 |

---
